# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 1: Data Loading & Understanding

**Objective:** Load the raw dataset and understand its structure, quality, and key characteristics before any cleaning or analysis.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [ ]:
df = pd.read_csv("../Data/advanced_cannibalization_dataset.csv")

print(f"Dataset loaded successfully!")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")

## 2. Dataset Structure

In [ ]:
# Column names and data types
print("Column Names & Data Types:\n")
print(df.dtypes)

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Last 5 rows
df.tail()

## 3. Missing Value Analysis

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    'Missing Count': null_counts,
    'Missing %': null_pct
})
print(null_report)

if null_counts.sum() == 0:
    print("\nNo missing values found. Dataset is complete.")
else:
    print(f"\nTotal missing values: {null_counts.sum()}")

## 4. Duplicate Records

In [ ]:
dupes = df.duplicated().sum()

print(f"Duplicate rows: {dupes}")
if dupes == 0:
    print("No duplicate records found.")
else:
    print(f"{dupes} duplicates detected — will handle in Phase 2.")

## 5. Descriptive Statistics

In [ ]:
num_cols = ['Price', 'Rating', 'Marketing_Spend', 'Stock_Available', 'Sales', 'Revenue']
df[num_cols].describe().round(2)

## 6. Categorical Variables

In [ ]:
# Unique Products
print(f"Unique Products: {df['Product_ID'].nunique()}")
print(sorted(df['Product_ID'].unique()))

# Category Distribution
print(f"\nCategory Distribution:")
print(df['Category'].value_counts().to_string())

# Product Group → Product Mapping
print(f"\nProduct Group → Products:")
pg_map = df.groupby('Product_Group')['Product_ID'].apply(lambda x: sorted(x.unique()))
for grp, prods in pg_map.items():
    print(f"  {grp}: {prods}")

# Region Distribution
print(f"\nRegion Distribution:")
print(df['Region'].value_counts().to_string())

# Launch Flag
print(f"\nLaunch Flag (1 = Launch Period):")
print(df['Launch_Flag'].value_counts().to_string())

## 7. Date Range Analysis

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

print(f"Earliest Date : {df['Date'].min().date()}")
print(f"Latest Date   : {df['Date'].max().date()}")
print(f"Date Span     : {(df['Date'].max() - df['Date'].min()).days} days")

# Monthly sales range
monthly = df.groupby(df['Date'].dt.to_period('M'))['Sales'].sum()
print(f"\nMonthly Sales Range: {monthly.min():,} – {monthly.max():,} units")

## 8. Launched Products Identification

In [ ]:
launched = df[df['Launch_Flag'] == 1].groupby('Product_ID').agg(
    Launch_Start = ('Date', 'min'),
    Launch_End   = ('Date', 'max'),
    Records      = ('Launch_Flag', 'count')
).reset_index()

print(f"Products with Launch_Flag = 1: {launched['Product_ID'].nunique()}")
print(launched.to_string(index=False))

print("\nLaunch Window: 2024-06-01 onwards")
print("Launched Products: P1, P2, P3 (Group G1) | P4, P5 (Group G2)")

## 9. Stock Availability

In [ ]:
stock_dist = df['Stock_Available'].value_counts()
print("Stock Availability:")
print(f"  In Stock (1)     : {stock_dist.get(1, 0):,}")
print(f"  Out of Stock (0) : {stock_dist.get(0, 0):,}")

print(f"\nOut-of-stock records per product:")
oos = df[df['Stock_Available'] == 0].groupby('Product_ID').size().sort_values(ascending=False)
print(oos.to_string())

## 10. Phase 1 Summary

In [ ]:
print("=" * 50)
print("  PHASE 1 SUMMARY")
print("=" * 50)
print(f"  Total Records      : {len(df):,}")
print(f"  Total Columns      : {df.shape[1]}")
print(f"  Products           : {df['Product_ID'].nunique()}")
print(f"  Product Groups     : {df['Product_Group'].nunique()}")
print(f"  Categories         : {df['Category'].nunique()}")
print(f"  Regions            : {df['Region'].nunique()}")
print(f"  Date Range         : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"  Missing Values     : {df.isnull().sum().sum()}")
print(f"  Duplicates         : {df.duplicated().sum()}")
print(f"  Launched Products  : P1, P2, P3, P4, P5 (from 2024-06-01)")
print(f"  Out-of-Stock Rows  : {(df['Stock_Available']==0).sum():,}")
print(f"  Total Revenue      : {df['Revenue'].sum():,.0f}")
print(f"  Total Units Sold   : {df['Sales'].sum():,}")
print("\nPhase 1 Complete. Proceed to Phase 2.")